In [ ]:
%load_ext autoreload
%autoreload 2

import glob
import itertools
import os
import pickle
import sys

import dotenv
import numpy as np
import pandas as pd

dotenv.load_dotenv()
PROJECT_ROOT = os.environ["PROJECT_ROOT"]
sys.path.append(PROJECT_ROOT)

import mpl.experiments.utils as OPE_utils

In [ ]:
RESULTS = {}

In [ ]:
Ks = [5, 10, 20, 30]
FOLDS = [1, 2, 3, 4, 5]
ls = {
    "0.1": 1e-1,
    "0.01": 1e-2,
    "0.001": 1e-3,
    "0.0001": 1e-4,
    "1e-05": 1e-5,
    "1e-06": 1e-6,
}
dataset = "mslr"
result_path = f"{PROJECT_ROOT}/experiments"
for K, FOLD, (l_str, l_val) in itertools.product(Ks, FOLDS, ls.items()):
    print(K, FOLD, l_str)
    prop_paths = {
        "MPL": glob.glob(
            f"{PROJECT_ROOT}/experiments/propensities/{dataset}/{FOLD}/RQ12/MPL/K={K};N=*0;l={l_str};seed=*;subset=test/propensities.npy"
        ),
        "MC": glob.glob(
            f"{PROJECT_ROOT}/experiments/propensities/{dataset}/{FOLD}/RQ12/MC/K={K};N=100*;seed=*;subset=test/propensities.npy"
        ),
    }
    time_paths = {
        "MPL": glob.glob(
            f"{PROJECT_ROOT}/experiments/propensities/{dataset}/{FOLD}/RQ12/MPL/K={K};N=*0;l={l_str};seed=*;subset=test/time.npy"
        ),
        "MC": glob.glob(
            f"{PROJECT_ROOT}/experiments/propensities/{dataset}/{FOLD}/RQ12/MC/K={K};N=100*;seed=*;subset=test/time.npy"
        ),
    }
    mappings = {
        7: ["-", "dataset", "FOLD", "model", "method", "pol"],  # IPM
    }
    if len(prop_paths["MPL"]) == 0:
        continue
    records = []
    ps = []
    for path, time_path in (
        (y1, y2)
        for x1, x2 in zip(prop_paths.values(), time_paths.values())
        for y1, y2 in zip(sorted(x1), sorted(x2))
    ):
        propensities = np.load(path).astype(dtype=np.float32)
        time = np.load(time_path)
        params = OPE_utils.flatten_dict(
            OPE_utils.path_to_params(path, result_path, mappings), sep=":"
        )
        if (params["pol:N"] == 1000) and (params["method"] == "MPL"):
            mask = propensities[:, 0, :, 0] != 0
            masks = {"mslr": mask}
        records.append({**params, "p": propensities[..., :K], "t": time.item()})
    df = pd.DataFrame.from_records(records)
    min_len = df["p"].apply(lambda x: x.shape[0]).min()
    df["p"] = df["p"].apply(lambda x: x[:min_len])
    masks["mslr"] = masks["mslr"][:min_len]
    df["pol:K"] = K
    df = df.sort_values(["dataset", "FOLD", "model", "pol:K", "pol:subset", "pol:N"])
    og_cols = [x for x in df.columns if x != "p"]
    ks = [K]
    k_masks = {
        (k, ds): (m[..., None] & (m.sum(-1)[..., None, None] > np.arange(k)))
        for ds, m in masks.items()
        for k in ks  # for f, m in enumerate(M)
    }
    max_idx = {"mslr": {1: 5816, 2: 5829, 3: 5821, 4: 5789, 5: 5854}}
    max_index = max_idx[dataset][FOLD]
    ref_cols = ["dataset", "FOLD", "model", "pol:K", "pol:subset"]
    preds = (
        df[(df["method"] == "MPL") & (df["pol:N"] == 1000)]["p"]
        .apply(lambda x: np.log(x[:, 0, :, 0]))
        .iloc[:1]
    )
    mins = preds.apply(lambda x: np.min(np.where(np.isinf(x), np.inf, x), -1))
    maxs = preds.apply(lambda x: np.max(x, -1))
    spreads = maxs - mins
    oracle = df[df["pol:N"] == 10000001]["p"].iloc[0][: max_idx[dataset][FOLD]]
    oracle_not_0 = oracle != 0
    df["p"] = df["p"].apply(lambda x: np.maximum(x, 1 / 10000001))
    df["p"] = df.apply(lambda x: x["p"][: max_idx[x["dataset"]][FOLD]], axis=1)

    # extract the reference arrays where D=True
    ref_E = df.loc[df["pol:N"] == 10000001, ref_cols + ["p"]].rename(
        columns={"p": "p_ref"}
    )

    # merge  back to original df so each row gets its group's reference array
    df = df.merge(ref_E, on=ref_cols, how="left")

    MAE = lambda y_true, y_pred: np.abs(y_true - y_pred)
    sMAPE = lambda y_true, y_pred: MAE(y_true, y_pred) / (
        (np.abs(y_true) + np.abs(y_pred)) * 1
    )
    MSE = lambda y_true, y_pred: (y_true - y_pred) ** 2
    metrics = {"sMAPE": sMAPE, "MSE": MSE}
    for metric, fn in metrics.items():
        df[metric] = [fn(e_ref, e) for e, e_ref in zip(df["p"], df["p_ref"])]
    df["pol:l"] = df["pol:l"].fillna(0)

    n_repeats = 1
    for i, title in enumerate(["num_docs", "logit_spread", "prop_value"]):
        if i == 0:
            quantiles = np.array([0, 1 / 3, 2 / 3, 1])
            thresholds = {}
            for ds, mask in masks.items():
                counts = mask.sum(axis=-1)
                quantile_values = np.quantile(counts, quantiles)
                thresholds[ds] = dict(zip(quantiles, quantile_values))
                print(f"{ds}: {quantile_values}")

            k_quantile_masks = {}
            for (k, ds), mask in k_masks.items():
                if ds == "yahoo":
                    continue
                for i in range(len(thresholds[ds]) - 1):
                    t, tp1 = (
                        list(thresholds[ds].values())[i],
                        list(thresholds[ds].values())[i + 1],
                    )
                    quantile = list(thresholds[ds].keys())[i]
                    k_quantile_masks[(k, ds, quantile)] = (
                        mask[:, None]
                        & (
                            (mask[..., 0].sum(-1, keepdims=True) >= t)
                            & (mask[..., 0].sum(-1, keepdims=True) <= tp1)
                        )[:, None, None]
                    )
        elif i == 1:
            quantiles = np.array([0, 0.05, 0.5, 1])
            thresholds = {}
            for ds, spread in spreads.items():
                if ds == 0:
                    ds = "mslr"
                quantile_values = np.quantile(spread, quantiles)
                thresholds[ds] = dict(zip(quantiles, quantile_values))
                print(f"{ds}: {quantile_values}")
            k_quantile_masks = {}
            for (k, ds), mask in k_masks.items():
                if ds == "yahoo":
                    continue
                for i in range(len(thresholds[ds]) - 1):
                    t, tp1 = (
                        list(thresholds[ds].values())[i],
                        list(thresholds[ds].values())[i + 1],
                    )
                    quantile = list(thresholds[ds].keys())[i]
                    k_quantile_masks[(k, ds, quantile)] = np.tile(
                        (
                            mask
                            & ((spreads.iloc[0] >= t) & (spreads.iloc[0] <= tp1))[
                                :, None, None
                            ]
                        )[:, None],
                        (1, n_repeats, 1, 1),
                    )
        elif i == 2:
            df["props"] = df["p"].apply(
                lambda x: x[:max_index][
                    np.tile(k_masks[(K, "mslr")][:, None], (1, n_repeats, 1, 1))
                    & oracle_not_0
                ]
            )
            q = df[df["pol:N"] == 10000001]["props"].iloc[0]
            quantiles = np.array([0, 1 / 3, 2 / 3, 1])
            thresholds = {}
            ds = "mslr"
            quantile_values = np.quantile(q, quantiles)
            thresholds[ds] = dict(zip(quantiles, quantile_values))
            print(f"{ds}: {quantile_values}")

            k_quantile_masks = {}
            for (k, ds), mask in k_masks.items():
                if ds == "yahoo":
                    continue
                for i in range(len(thresholds[ds]) - 1):
                    t, tp1 = (
                        list(thresholds[ds].values())[i],
                        list(thresholds[ds].values())[i + 1],
                    )
                    quantile = list(thresholds[ds].keys())[i]
                    k_quantile_masks[(k, ds, quantile)] = mask[:, None] & (
                        (oracle >= t) & (oracle <= tp1)
                    )
        _metrics = [f"_{metric}" for metric in metrics]
        _mqs = []
        for j, q in enumerate(quantiles[:-1]):
            for metric in metrics:
                mq = f"_{metric}_{q:.2f}"
                df[mq] = df.apply(
                    lambda x: np.ma.masked_array(
                        x[metric],
                        ~(
                            k_quantile_masks[(x["pol:K"], x["dataset"], q)]
                            & oracle_not_0
                        ),
                    )
                    .mean((0, 2, 3))
                    .data,
                    axis=1,
                )
                _mqs.append(mq)
        for metric in metrics:
            mq = f"_{metric}"
            df[mq] = df.apply(
                lambda x: np.ma.masked_array(
                    x[metric],
                    ~(
                        np.tile(
                            k_masks[(x["pol:K"], x["dataset"])][:, None],
                            (1, n_repeats, 1, 1),
                        )
                        & oracle_not_0
                    ),
                )
                .mean((0, 2, 3))
                .data,
                axis=1,
            )
            _mqs.append(mq)

        for metric in metrics:
            rel_cols = [x for x in _mqs if x.startswith(f"_{metric}")]
            for method, _group in df.groupby("method"):
                for j, (key, group) in enumerate(_group.groupby("pol:l")):
                    x = group[group["pol:N"] != 10000001]["t"]
                    for i, col in enumerate(rel_cols):
                        y = group[group["pol:N"] != 10000001][col]
                        KEY = {
                            "dataset": dataset,
                            "fold": FOLD,
                            "x": "time",
                            "y": col,
                            "K": K,
                            "method": method,
                            "l": key,
                            "groupby": title,
                        }
                        RESULTS[OPE_utils.params_to_path(KEY)] = {
                            "y": (np.vstack(y.values).T if len(y.values) else None),
                            "x": x.values,
                        }

In [ ]:
# Save the results
plotting_result_path = os.path.join(result_path, "results")
os.makedirs(plotting_result_path, exist_ok=True)
pickle.dump(RESULTS, open(f"{plotting_result_path}/test_results.pkl", "wb"))

In [ ]:
# Load the results
data_path = os.path.join(plotting_result_path, "test_results.pkl")
figure_dir = os.path.join(plotting_result_path, "figures")
os.makedirs(figure_dir, exist_ok=True)

In [ ]:
def maybe_float(s: str) -> bool | str:
    try:
        return float(s)
    except ValueError:
        return s


def str_to_d(x, sep=";"):
    return dict([[maybe_float(z) for z in y.split("=")] for y in x.split(sep)])


data = pickle.load(open(data_path, "rb"))
data = [(str_to_d(k), v) for k, v in data.items()]
data = [{**a, "_x": b["x"], "_y": b["y"]} for a, b in data]
df = pd.DataFrame.from_records(data)
df["_x"] = df["_x"] / df["fold"].apply(
    lambda x: (max_idx["mslr"][x] - 2 * 16) / 1000
)  # Calculate per-query time

# Use average per-query time for all runs with the same expected computational cost
average_times = df.groupby(["method", "K"])["_x"].apply(
    lambda x: np.stack(np.array(x)).mean(0)
)
df = df.drop(["_x"], axis=1).merge(average_times, on=["method", "K"], how="left")


def stack_and_agg(subdf):
    # handle c
    mat_list = [np.atleast_2d(x) for x in subdf["_y"]]
    stacked = np.concatenate(mat_list, axis=0)

    # aggregate other columns
    _x = subdf["_x"].iloc[0]

    return pd.Series(
        {
            "_x": _x,
            "_y_mean": stacked.mean(axis=0),
            "_y_max": stacked.max(axis=0),
            "_y_min": stacked.min(axis=0),
        }
    )


df = df.groupby(["K", "l", "method", "y", "groupby"]).apply(stack_and_agg).reset_index()

In [ ]:
xcoef = 5800 * 5 / 1000
global_ylim = {"MSE": (7e-14, 5e-5), "sMAPE": (1e-3, 1.03)}
global_ylim = {"MSE": (7e-14, 5e-5), "sMAPE": (1e-3, 1)}
global_xlim = {
    "MSE": (4 / xcoef, 12000 / xcoef),
    "sMAPE": (4 / xcoef, 12000 / xcoef),
}
figsize = (1.5, 1.5)

In [ ]:
import pylab as plt

plt.style.use("default")
plt.rc("font", size=12, family="serif")
plt.rc("text", usetex=True)
plt.rc("text.latex", preamble=r"\usepackage{amsmath,amssymb,bm,bbm,lmodern}")

In [ ]:
# N vs MSE, split by |D| (color) and method (linestyle)
linestyles = {"MPL": "-", "MC": "--"}
metric = "MSE"
split = "num_docs"
K = 10
quantiles = ["0.00", "0.33", "0.67"]
colors = {
    f"_{metric}_0.00": "C0",
    f"_{metric}_0.33": "C1",
    f"_{metric}_0.67": "C2",
    f"_{metric}": "C3",
}
labels = {
    f"_{metric}_0.00": r"$~~|D|\leq 87$",
    f"_{metric}_0.33": r"$87 < |D|\leq 130$",
    f"_{metric}_0.67": r"$130 < |D|\leq 250$",
    f"_{metric}": "all",
}
zorders = {
    f"_{metric}_0.00": 0,
    f"_{metric}_0.33": 0,
    f"_{metric}_0.67": 2,
    f"_{metric}": +3,
}
markerparams = {
    "MPL": {"marker": "v", "markersize": 5},
    "MC": {"marker": "o", "markersize": 5},
}

groupby = ["method", "y"]
line = "mean"
ci = "minmax"
mask = (
    (df["K"] == K)
    & df["y"].apply(lambda x: x.startswith(f"_{metric}"))
    & df["l"].apply(lambda x: x in [0, 1e-5])
    & (df["groupby"] == split)
)
_df = df[mask]

fig, ax = plt.subplots(figsize=figsize)
for g, d in _df.groupby(groupby):
    linestyle = linestyles[g[0]]
    color = colors[g[1]]
    label = f"{g[0]} {labels[g[1]]}"
    zorder = zorders[g[1]]
    markerparam = markerparams[g[0]]
    x, y = d[f"_x"].item(), d[f"_y_{line}"].item()
    y_low, y_high = {"minmax": (d["_y_min"].item(), d["_y_max"].item())}[ci]
    ax.plot(
        x,
        y,
        linestyle=linestyle,
        color=color,
        label=label,
        zorder=zorder,
        **markerparam,
    )
    ax.fill_between(x, y_low, y_high, color=color, alpha=0.5, zorder=zorder)
    ax.set_yscale("log")
    ax.set_xscale("log")
    ax.set_yticklabels([])
    if global_ylim.get(metric, None) is not None:
        ax.set_ylim(global_ylim[metric])
    if global_xlim.get(metric, None) is not None:
        ax.set_xlim(global_xlim[metric])
fig.patch.set_facecolor("white")

ax.set_facecolor("white")
ax.tick_params(axis="x", colors="black")  # Set tick color if needed
ax.tick_params(axis="y", colors="black")  # Set tick color if needed
figure_path = f"{figure_dir}/metric={metric};split={split};K={K}.pdf"
fig.savefig(figure_path, bbox_inches="tight", dpi=300, pad_inches=0.02)

In [ ]:
# N vs sMAPE, split by |D| (color) and method (linestyle)
linestyles = {"MPL": "-", "MC": "--"}
metric = "sMAPE"
split = "num_docs"
K = 10
quantiles = ["0.00", "0.33", "0.67"]
colors = {
    f"_{metric}_0.00": "C0",
    f"_{metric}_0.33": "C1",
    f"_{metric}_0.67": "C2",
    f"_{metric}": "C3",
}
labels = {
    f"_{metric}_0.00": r"$\quad\; 0<|D|\leq 87$",
    f"_{metric}_0.33": r"$\;\:\,87< |D|\leq 130$",
    f"_{metric}_0.67": r"$130< |D|\leq 250$",
    f"_{metric}": r"$\qquad\quad \text{all}$",
}
zorders = {
    f"_{metric}_0.00": 0,
    f"_{metric}_0.33": 0,
    f"_{metric}_0.67": 2,
    f"_{metric}": +3,
}
markerparams = {
    "MPL": {"marker": "v", "markersize": 5},
    "MC": {"marker": "o", "markersize": 5},
}

groupby = ["method", "y"]
line = "mean"
ci = "minmax"
mask = (
    (df["K"] == K)
    & df["y"].apply(lambda x: x.startswith(f"_{metric}"))
    & df["l"].apply(lambda x: x in [0, 1e-5])
    & (df["groupby"] == split)
)
_df = df[mask]

fig, ax = plt.subplots(figsize=figsize)
for g, d in _df.groupby(groupby):
    linestyle = linestyles[g[0]]
    color = colors[g[1]]
    label = f"{g[0]} {labels[g[1]]}"
    zorder = zorders[g[1]]
    markerparam = markerparams[g[0]]
    x, y = d[f"_x"].item(), d[f"_y_{line}"].item()
    y_low, y_high = {"minmax": (d["_y_min"].item(), d["_y_max"].item())}[ci]
    ax.plot(
        x,
        y,
        linestyle=linestyle,
        color=color,
        label=label,
        zorder=zorder,
        **markerparam,
    )
    ax.fill_between(x, y_low, y_high, color=color, alpha=0.5, zorder=zorder)
    ax.set_yscale("log")
    ax.set_xscale("log")
    ax.set_yticklabels([])
    if global_ylim.get(metric, None) is not None:
        ax.set_ylim(global_ylim[metric])
    if global_xlim.get(metric, None) is not None:
        ax.set_xlim(global_xlim[metric])
figure_path = f"{figure_dir}/metric={metric};split={split};K={K}.pdf"
fig.savefig(figure_path, bbox_inches="tight", dpi=300, pad_inches=0.02)

In [ ]:
import matplotlib.lines as mlines

legend_info = []
for k in colors:
    legend_info.append(
        {
            "linestyle": "-",
            "color": colors[k],
            "marker": "s",
            "markersize": 12,
            "markevery": 1,
            "fillstyle": "full",
            "label": labels[k],
            "linewidth": None,
        }
    )

ncol = 1
figlegend = plt.figure(figsize=(0.5, 0.5))
a = figlegend.legend(
    handles=[mlines.Line2D([], [], **l) for l in legend_info],
    fontsize=18,
    loc="center",
    ncol=ncol,
    frameon=False,
    borderaxespad=0,
    borderpad=0.3,
    labelspacing=0.0,
    columnspacing=1,
    handlelength=0.1,
)
offsets = {
    f"_{metric}_0.00": 16.45,
    f"_{metric}_0.33": 8.22,
    f"_{metric}_0.67": 0,
    f"_{metric}": 52,
}
for t, offset in zip(a.get_texts(), offsets.values()):
    t.set_position((offset, 0))
figlegend.savefig(
    f"{figure_dir}/legend_metric={metric};split={split};K={K}.pdf",
    bbox_inches="tight",
    pad_inches=0.02,
)

In [ ]:
# N vs MSE, split by spread (color) and method (linestyle)
linestyles = {"MPL": "-", "MC": "--"}
metric = "MSE"
split = "logit_spread"
K = 10
quantiles = ["0.00", "0.33", "0.67"]
colors = {
    f"_{metric}_0.00": "C0",
    f"_{metric}_0.05": "C1",
    f"_{metric}_0.50": "C2",
    f"_{metric}": "C3",
}
labels = {
    f"_{metric}_0.00": r"$\Delta\leq 12.6$",
    f"_{metric}_0.05": r"$12.6 < \Delta \leq 29.2$",
    f"_{metric}_0.50": r"$29.2 < \Delta \leq 40$",
    f"_{metric}": "all",
}
zorders = {
    f"_{metric}_0.00": 0,
    f"_{metric}_0.05": 0,
    f"_{metric}_0.50": 2,
    f"_{metric}": +3,
}
markerparams = {
    "MPL": {"marker": "v", "markersize": 5},
    "MC": {"marker": "o", "markersize": 5},
}

groupby = ["method", "y"]
line = "mean"
ci = "minmax"
mask = (
    (df["K"] == K)
    & df["y"].apply(lambda x: x.startswith(f"_{metric}"))
    & df["l"].apply(lambda x: x in [0, 1e-5])
    & (df["groupby"] == split)
)
_df = df[mask]

fig, ax = plt.subplots(figsize=figsize)
for g, d in _df.groupby(groupby):
    linestyle = linestyles[g[0]]
    color = colors[g[1]]
    label = f"{g[0]} {labels[g[1]]}"
    zorder = zorders[g[1]]
    markerparam = markerparams[g[0]]
    x, y = d[f"_x"].item(), d[f"_y_{line}"].item()
    y_low, y_high = {"minmax": (d["_y_min"].item(), d["_y_max"].item())}[ci]
    ax.plot(
        x,
        y,
        linestyle=linestyle,
        color=color,
        label=label,
        zorder=zorder,
        **markerparam,
    )
    ax.fill_between(x, y_low, y_high, color=color, alpha=0.5, zorder=zorder)
    ax.set_yscale("log")
    ax.set_xscale("log")
    if global_ylim.get(metric, None) is not None:
        ax.set_ylim(global_ylim[metric])
    if global_xlim.get(metric, None) is not None:
        ax.set_xlim(global_xlim[metric])
ax.set_yticklabels([])
figure_path = f"{figure_dir}/metric={metric};split={split};K={K}.pdf"
fig.savefig(figure_path, bbox_inches="tight", dpi=300, pad_inches=0.02)

In [ ]:
# N vs sMAPE, split by spread (color) and method (linestyle)
linestyles = {"MPL": "-", "MC": "--"}
metric = "sMAPE"
split = "logit_spread"
K = 10
quantiles = ["0.00", "0.33", "0.67"]
colors = {
    f"_{metric}_0.00": "C0",
    f"_{metric}_0.05": "C1",
    f"_{metric}_0.50": "C2",
    f"_{metric}": "C3",
}
labels = {
    f"_{metric}_0.00": r"$\quad\;\;\:\! 0 <  \Delta\leq 12.6$",
    f"_{metric}_0.05": r"$12.6 < \Delta \leq 29.2$",
    f"_{metric}_0.50": r"$\;\!29.2 < \Delta \leq 39.3$",
    f"_{metric}": r"$\qquad\quad\;$all",
}
zorders = {
    f"_{metric}_0.00": 0,
    f"_{metric}_0.05": 0,
    f"_{metric}_0.50": 2,
    f"_{metric}": +3,
}
markerparams = {
    "MPL": {"marker": "v", "markersize": 5},
    "MC": {"marker": "o", "markersize": 5},
}

groupby = ["method", "y"]
line = "mean"
ci = "minmax"
mask = (
    (df["K"] == K)
    & df["y"].apply(lambda x: x.startswith(f"_{metric}"))
    & df["l"].apply(lambda x: x in [0, 1e-5])
    & (df["groupby"] == split)
)
_df = df[mask]

fig, ax = plt.subplots(figsize=figsize)
for g, d in _df.groupby(groupby):
    linestyle = linestyles[g[0]]
    color = colors[g[1]]
    label = f"{g[0]} {labels[g[1]]}"
    zorder = zorders[g[1]]
    markerparam = markerparams[g[0]]
    x, y = d[f"_x"].item(), d[f"_y_{line}"].item()
    y_low, y_high = {"minmax": (d["_y_min"].item(), d["_y_max"].item())}[ci]
    ax.plot(
        x,
        y,
        linestyle=linestyle,
        color=color,
        label=label,
        zorder=zorder,
        **markerparam,
    )
    ax.fill_between(x, y_low, y_high, color=color, alpha=0.5, zorder=zorder)
    ax.set_yscale("log")
    ax.set_xscale("log")
    if global_ylim.get(metric, None) is not None:
        ax.set_ylim(global_ylim[metric])
    if global_xlim.get(metric, None) is not None:
        ax.set_xlim(global_xlim[metric])
ax.set_yticklabels([])
figure_path = f"{figure_dir}/metric={metric};split={split};K={K}.pdf"
fig.savefig(figure_path, bbox_inches="tight", dpi=300, pad_inches=0.02)

In [ ]:
import matplotlib.lines as mlines

legend_info = []
for k in colors:
    legend_info.append(
        {
            "linestyle": "-",
            "color": colors[k],
            "marker": "s",
            "markersize": 12,
            "markevery": 1,
            "fillstyle": "full",
            "label": labels[k],
            "linewidth": None,
        }
    )

ncol = 1
figlegend = plt.figure(figsize=(0.5, 0.5))
a = figlegend.legend(
    handles=[mlines.Line2D([], [], **l) for l in legend_info],
    fontsize=18,
    loc="center",
    ncol=ncol,
    frameon=False,
    borderaxespad=0,
    borderpad=0.3,
    labelspacing=0.0,
    columnspacing=1,
    handlelength=0.1,
)
offsets = {
    f"_{metric}_0.00": 21.3,
    f"_{metric}_0.33": 0,
    f"_{metric}_0.67": 0,
    f"_{metric}": 51.5,
}
for t, offset in zip(a.get_texts(), offsets.values()):
    t.set_position((offset, 0))
figlegend.savefig(
    f"{figure_dir}/legend_metric={metric};split={split};K={K}.pdf",
    bbox_inches="tight",
    pad_inches=0.02,
)

In [ ]:
# N vs MSE, split by prop value (color) and method (linestyle)
linestyles = {"MPL": "-", "MC": "--"}
metric = "MSE"
split = "prop_value"
K = 10
quantiles = ["0.00", "0.33", "0.67"]
colors = {
    f"_{metric}_0.00": "C0",
    f"_{metric}_0.33": "C1",
    f"_{metric}_0.67": "C2",
    f"_{metric}": "C3",
}
labels = {
    f"_{metric}_0.00": r"$\hat{p}\leq 5e-6$",
    f"_{metric}_0.33": r"$5e-6 < \hat{p} < 4e-4$",
    f"_{metric}_0.67": r"$4e-4 < \hat{p}$",
    f"_{metric}": r"$\quad\;$all",
}
zorders = {
    f"_{metric}_0.00": 0,
    f"_{metric}_0.33": 0,
    f"_{metric}_0.67": 2,
    f"_{metric}": +3,
}
markerparams = {
    "MPL": {"marker": "v", "markersize": 5},
    "MC": {"marker": "o", "markersize": 5},
}

groupby = ["method", "y"]
line = "mean"
ci = "minmax"
mask = (
    (df["K"] == K)
    & df["y"].apply(lambda x: x.startswith(f"_{metric}"))
    & df["l"].apply(lambda x: x in [0, 1e-5])
    & (df["groupby"] == split)
)
_df = df[mask]

fig, ax = plt.subplots(figsize=figsize)
for g, d in _df.groupby(groupby):
    linestyle = linestyles[g[0]]
    color = colors[g[1]]
    label = f"{g[0]} {labels[g[1]]}"
    zorder = zorders[g[1]]
    markerparam = markerparams[g[0]]
    x, y = d[f"_x"].item(), d[f"_y_{line}"].item()
    y_low, y_high = {"minmax": (d["_y_min"].item(), d["_y_max"].item())}[ci]
    ax.plot(
        x,
        y,
        linestyle=linestyle,
        color=color,
        label=label,
        zorder=zorder,
        **markerparam,
    )
    ax.fill_between(x, y_low, y_high, color=color, alpha=0.5, zorder=zorder)
    ax.set_yscale("log")
    ax.set_xscale("log")
    if global_ylim.get(metric, None) is not None:
        ax.set_ylim(global_ylim[metric])
    if global_xlim.get(metric, None) is not None:
        ax.set_xlim(global_xlim[metric])
ax.set_yticklabels([])
figure_path = f"{figure_dir}/metric={metric};split={split};K={K}.pdf"
fig.savefig(figure_path, bbox_inches="tight", dpi=300, pad_inches=0.02)

In [ ]:
# N vs sMAPE, split by prop value (color) and method (linestyle)
linestyles = {"MPL": "-", "MC": "--"}
metric = "sMAPE"
split = "prop_value"
K = 10
quantiles = ["0.00", "0.33", "0.67"]
colors = {
    f"_{metric}_0.00": "C0",
    f"_{metric}_0.33": "C1",
    f"_{metric}_0.67": "C2",
    f"_{metric}": "C3",
}
labels = {
    f"_{metric}_0.00": r"$\quad\,\, 10^{-7} \leq p \leq 5\cdot 10^{-6}$",
    f"_{metric}_0.33": r"$5\cdot 10^{-6} < p \leq 4\cdot 10^{-4}$",
    f"_{metric}_0.67": r"$\:\!4\cdot 10^{-4} < p \leq 1$",
    f"_{metric}": r"$\qquad\qquad\;\;\:$all",
}
zorders = {
    f"_{metric}_0.00": 0,
    f"_{metric}_0.33": 0,
    f"_{metric}_0.67": 2,
    f"_{metric}": +3,
}
markerparams = {
    "MPL": {"marker": "v", "markersize": 5},
    "MC": {"marker": "o", "markersize": 5},
}

groupby = ["method", "y"]
line = "mean"
ci = "minmax"
mask = (
    (df["K"] == K)
    & df["y"].apply(lambda x: x.startswith(f"_{metric}"))
    & df["l"].apply(lambda x: x in [0, 1e-5])
    & (df["groupby"] == split)
)
_df = df[mask]

fig, ax = plt.subplots(figsize=figsize)
for g, d in _df.groupby(groupby):
    linestyle = linestyles[g[0]]
    color = colors[g[1]]
    label = f"{g[0]} {labels[g[1]]}"
    zorder = zorders[g[1]]
    markerparam = markerparams[g[0]]
    x, y = d[f"_x"].item(), d[f"_y_{line}"].item()
    y_low, y_high = {"minmax": (d["_y_min"].item(), d["_y_max"].item())}[ci]
    ax.plot(
        x,
        y,
        linestyle=linestyle,
        color=color,
        label=label,
        zorder=zorder,
        **markerparam,
    )
    ax.fill_between(x, y_low, y_high, color=color, alpha=0.5, zorder=zorder)
    ax.set_yscale("log")
    ax.set_xscale("log")
    if global_ylim.get(metric, None) is not None:
        ax.set_ylim(global_ylim[metric])
    if global_xlim.get(metric, None) is not None:
        ax.set_xlim(global_xlim[metric])
ax.set_yticklabels([])
figure_path = f"{figure_dir}/metric={metric};split={split};K={K}.pdf"
fig.savefig(figure_path, bbox_inches="tight", dpi=300, pad_inches=0.02)

In [ ]:
import matplotlib.lines as mlines

legend_info = []
for k in colors:
    legend_info.append(
        {
            "linestyle": "-",
            "color": colors[k],
            "marker": "s",
            "markersize": 12,
            "markevery": 1,
            "fillstyle": "full",
            "label": labels[k],
            "linewidth": None,
        }
    )

ncol = 1
figlegend = plt.figure(figsize=(0.5, 0.5))
a = figlegend.legend(
    handles=[mlines.Line2D([], [], **l) for l in legend_info],
    fontsize=18,
    loc="center",
    ncol=ncol,
    frameon=False,
    borderaxespad=0,
    borderpad=0.3,
    labelspacing=0.0,
    columnspacing=1,
    handlelength=0.1,
)
offsets = {
    f"_{metric}_0.00": 20.9,
    f"_{metric}_0.33": 0,
    f"_{metric}_0.67": 0,
    f"_{metric}": 73,
}
for t, offset in zip(a.get_texts(), offsets.values()):
    t.set_position((offset, 0))
figlegend.savefig(
    f"{figure_dir}/legend_metric={metric};split={split};K={K}.pdf",
    bbox_inches="tight",
    pad_inches=0.02,
)

In [ ]:
# Us: N vs MSE, split by K (color) and method (linestyle)
linestyles = {"MPL": "-", "MC": "--"}
metric = "MSE"
split = "prop_value"
colors = {5: "C0", 10: "C3", 20: "C1", 30: "C2"}
labels = {5: r"$K=5$", 10: r"$K=10$", 20: r"$K=20$", 30: r"$K=30$"}
zorders = {5: 0, 10: +3, 20: 1, 30: 2}
markerparams = {
    "MPL": {"marker": "v", "markersize": 5},
    "MC": {"marker": "o", "markersize": 5},
}

groupby = ["method", "K"]
line = "mean"
ci = "minmax"
mask = (
    df["y"].apply(lambda x: (x == f"_{metric}"))
    & df["l"].apply(lambda x: x in [0, 1e-5])
    & (df["groupby"] == split)
)
_df = df[mask]

fig, ax = plt.subplots(figsize=figsize)
for g, d in _df.groupby(groupby):
    linestyle = linestyles[g[0]]
    color = colors[g[1]]
    label = f"{g[0]} {labels[g[1]]}"
    zorder = zorders[g[1]]
    markerparam = markerparams[g[0]]
    x, y = d[f"_x"].item(), d[f"_y_{line}"].item()
    y_low, y_high = {"minmax": (d["_y_min"].item(), d["_y_max"].item())}[ci]
    ax.plot(
        x,
        y,
        linestyle=linestyle,
        color=color,
        label=label,
        zorder=zorder,
        **markerparam,
    )
    ax.fill_between(x, y_low, y_high, color=color, alpha=0.5, zorder=zorder)
    ax.set_yscale("log")
    ax.set_xscale("log")
    if global_ylim.get(metric, None) is not None:
        ax.set_ylim(global_ylim[metric])
    if global_xlim.get(metric, None) is not None:
        ax.set_xlim(global_xlim[metric])
figure_path = f"{figure_dir}/metric={metric};split=K.pdf"
fig.savefig(figure_path, bbox_inches="tight", dpi=300, pad_inches=0.02)

In [ ]:
# Us: N vs sMAPE, split by K (color) and method (linestyle)
linestyles = {"MPL": "-", "MC": "--"}
metric = "sMAPE"
split = "prop_value"
colors = {5: "C0", 10: "C3", 20: "C1", 30: "C2"}
labels = {5: r"$K=5$", 10: r"$K=10$", 20: r"$K=20$", 30: r"$K=30$"}
zorders = {5: 0, 10: +3, 20: 1, 30: 2}
markerparams = {
    "MPL": {"marker": "v", "markersize": 5},
    "MC": {"marker": "o", "markersize": 5},
}

groupby = ["method", "K"]
line = "mean"
ci = "minmax"
mask = (
    df["y"].apply(lambda x: (x == f"_{metric}"))
    & df["l"].apply(lambda x: x in [0, 1e-5])
    & (df["groupby"] == split)
)
_df = df[mask]

fig, ax = plt.subplots(figsize=figsize)
for g, d in _df.groupby(groupby):
    linestyle = linestyles[g[0]]
    color = colors[g[1]]
    label = f"{g[0]} {labels[g[1]]}"
    zorder = zorders[g[1]]
    markerparam = markerparams[g[0]]
    x, y = d[f"_x"].item(), d[f"_y_{line}"].item()
    y_low, y_high = {"minmax": (d["_y_min"].item(), d["_y_max"].item())}[ci]
    ax.plot(
        x,
        y,
        linestyle=linestyle,
        color=color,
        label=label,
        zorder=zorder,
        **markerparam,
    )
    ax.fill_between(x, y_low, y_high, color=color, alpha=0.5, zorder=zorder)
    ax.set_yscale("log")
    ax.set_xscale("log")
    if global_ylim.get(metric, None) is not None:
        ax.set_ylim(global_ylim[metric])
    if global_xlim.get(metric, None) is not None:
        ax.set_xlim(global_xlim[metric])
figure_path = f"{figure_dir}/metric={metric};split=K.pdf"
fig.savefig(figure_path, bbox_inches="tight", dpi=300, pad_inches=0.02)

In [ ]:
import matplotlib.lines as mlines

legend_info = []
for k in colors:
    legend_info.append(
        {
            "linestyle": "-",
            "color": colors[k],
            "marker": "s",
            "markersize": 12,
            "markevery": 1,  # this should be higher if you have more data
            "fillstyle": "full",
            "label": labels[k],
            "linewidth": None,
        }
    )

ncol = 1
figlegend = plt.figure(figsize=(0.5, 0.5))
figlegend.legend(
    handles=[mlines.Line2D([], [], **l) for l in legend_info],
    fontsize=18,
    loc="center",
    ncol=ncol,
    frameon=False,
    borderaxespad=0,
    borderpad=0.3,
    labelspacing=0.0,
    columnspacing=1,
    handlelength=0.1,
)
figlegend.savefig(
    f"{figure_dir}/legend_metric={metric};split=K.pdf",
    bbox_inches="tight",
    pad_inches=0.02,
)

In [ ]:
K = 10

In [ ]:
# Us: N vs MSE, split by l (color), no bl
linestyles = {"MPL": "-"}
metric = "MSE"
split = "prop_value"
colors = {
    1e-1: "C0",
    1e-2: "C1",
    1e-3: "C2",
    1e-4: "C4",
    1e-5: "C3",
    1e-6: "C5",
    1e-7: "C6",
}
labels = {
    1e-1: "$1e-1$",
    1e-2: "$1e-2$",
    1e-3: "$1e-3$",
    1e-4: "$1e-4$",
    1e-5: "$1e-5$",
    1e-6: "$1e-6$",
    1e-7: "$1e-7$",
}
zorders = {1e-1: 0, 1e-2: 1, 1e-3: 2, 1e-4: 3, 1e-5: -4, 1e-6: 4, 1e-7: 5}
markerparams = {
    "MPL": {"marker": "v", "markersize": 5},
    "MC": {"marker": "o", "markersize": 5},
}

groupby = ["method", "l"]
line = "mean"
ci = "minmax"
mask = (
    (df["method"] == "MPL")
    & (df["K"] == 10)
    & df["y"].apply(lambda x: (x == f"_{metric}"))
    & (df["groupby"] == split)
)
_df = df[mask]

fig, ax = plt.subplots(figsize=(figsize[0] * 3, figsize[1]))
for g, d in _df.groupby(groupby):
    if g[1] == 1e-7:
        continue
    linestyle = linestyles[g[0]]
    color = colors[g[1]]
    label = f"{g[0]} {labels[g[1]]}"
    zorder = zorders[g[1]]
    markerparam = markerparams[g[0]]
    x, y = d[f"_x"].item()[:6], d[f"_y_{line}"].item()[:6]
    y_low, y_high = {"minmax": (d["_y_min"].item(), d["_y_max"].item())}[ci]
    ax.plot(
        x,
        y,
        linestyle=linestyle,
        color=color,
        label=label,
        zorder=zorder,
        **markerparam,
    )
    ax.fill_between(x, y_low, y_high, color=color, alpha=0.5, zorder=zorder)
    ax.set_yscale("log")
    ax.set_xscale("log")

from matplotlib.ticker import NullFormatter

ax.xaxis.set_minor_formatter(NullFormatter())
ax.set_xticks([2, 4, 8])
ax.set_xticklabels([2, 4, 8])
figure_path = f"{figure_dir}/metric={metric};split=l.pdf"
fig.savefig(figure_path, bbox_inches="tight", dpi=300, pad_inches=0.02)

In [ ]:
# Us: N vs sMAPE, split by l (color), no bl
linestyles = {"MPL": "-"}
metric = "sMAPE"
split = "prop_value"
colors = {1e-1: "C0", 1e-2: "C1", 1e-3: "C2", 1e-4: "C4", 1e-5: "C3", 1e-6: "C5"}
labels = {
    1e-1: "$1e-1$",
    1e-2: "$1e-2$",
    1e-3: "$1e-3$",
    1e-4: "$1e-4$",
    1e-5: "$1e-5$",
    1e-6: "$1e-6$",
}
labels = {
    1e-1: "$10^{-1}$",
    1e-2: "$10^{-2}$",
    1e-3: "$10^{-3}$",
    1e-4: "$10^{-4}$",
    1e-5: "$10^{-5}$",
    1e-6: "$10^{-6}$",
}
zorders = {1e-1: 0, 1e-2: 1, 1e-3: 2, 1e-4: 3, 1e-5: -4, 1e-6: 4}
markerparams = {
    "MPL": {"marker": "v", "markersize": 5},
    "MC": {"marker": "o", "markersize": 5},
}

groupby = ["method", "l"]
line = "mean"
ci = "minmax"
mask = (
    (df["method"] == "MPL")
    & (df["K"] == 10)
    & df["y"].apply(lambda x: (x == f"_{metric}"))
    & (df["groupby"] == split)
)
_df = df[mask]

fig, ax = plt.subplots(figsize=figsize)
for g, d in _df.groupby(groupby):
    if g[1] == 1e-7:
        continue
    linestyle = linestyles[g[0]]
    color = colors[g[1]]
    label = f"{g[0]} {labels[g[1]]}"
    zorder = zorders[g[1]]
    markerparam = markerparams[g[0]]
    x, y = d[f"_x"].item()[:6], d[f"_y_{line}"].item()[:6]
    y_low, y_high = {"minmax": (d["_y_min"].item(), d["_y_max"].item())}[ci]
    ax.plot(
        x,
        y,
        linestyle=linestyle,
        color=color,
        label=label,
        zorder=zorder,
        **markerparam,
    )
    ax.fill_between(x, y_low, y_high, color=color, alpha=0.5, zorder=zorder)
    ax.set_yscale("log")
    ax.set_xscale("log")
    minor_yticks = [0.055, 0.06]  # any list of positions
    from matplotlib.ticker import FixedLocator

    ax.yaxis.set_minor_locator(FixedLocator(minor_yticks))
    if global_xlim.get(metric, None) is not None:
        ax.set_xlim(global_xlim[metric])
ax.set_yticklabels([])
figure_path = f"{figure_dir}/metric={metric};split=l.pdf"
fig.savefig(figure_path, bbox_inches="tight", dpi=300, pad_inches=0.02)

In [ ]:
import matplotlib.lines as mlines

legend_info = []
for k in colors:
    legend_info.append(
        {
            "linestyle": "-",
            "color": colors[k],
            "marker": "v",
            "markersize": 12,
            "markevery": 1,  # this should be higher if you have more data
            "fillstyle": "full",
            "label": labels[k],
            "linewidth": None,
        }
    )

ncol = 2
figlegend = plt.figure(figsize=(0.5, 0.5))
figlegend.legend(
    handles=[mlines.Line2D([], [], **l) for l in legend_info],
    fontsize=18,
    loc="center",
    ncol=ncol,
    frameon=False,
    borderaxespad=0,
    borderpad=0.3,
    labelspacing=0.0,
    columnspacing=1,
    handlelength=0.1,
)
figlegend.savefig(
    f"{figure_dir}/legend_metric={metric};split=l.pdf",
    bbox_inches="tight",
    pad_inches=0.02,
)